# 📊 Análisis Exploratorio de Datos (EDA)
## Student Performance Factors Dataset

**Objetivo:** Comprender la estructura, distribución y relaciones del dataset
antes de proceder al preprocesamiento y modelado.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print("✅ Librerías importadas correctamente")

## 1. Carga y Exploración Inicial

In [ ]:
# Cargar dataset
df = pd.read_csv('../data/raw/StudentPerformanceFactors.csv')

print(f"📐 Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"\n📋 Tipos de datos:")
print(df.dtypes.value_counts())
print(f"\n🔍 Valores nulos:")
print(df.isnull().sum()[df.isnull().sum() > 0])
if df.isnull().sum().sum() == 0:
    print("✅ No hay valores nulos")

In [ ]:
# Vista previa
df.head(10)

In [ ]:
# Estadísticas descriptivas - numéricas
print("📊 Estadísticas descriptivas (numéricas):")
df.describe().round(2)

In [ ]:
# Estadísticas descriptivas - categóricas
print("📊 Estadísticas descriptivas (categóricas):")
df.describe(include='object')

## 2. Análisis de la Variable Objetivo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(df['Exam_Score'], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(df['Exam_Score'].mean(), color='red', linestyle='--', 
                label=f"Media: {df['Exam_Score'].mean():.1f}")
axes[0].axvline(df['Exam_Score'].median(), color='green', linestyle='--',
                label=f"Mediana: {df['Exam_Score'].median():.1f}")
axes[0].set_title('Distribución de Exam_Score', fontsize=14)
axes[0].set_xlabel('Exam Score')
axes[0].legend()

# Boxplot
axes[1].boxplot(df['Exam_Score'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[1].set_title('Boxplot de Exam_Score', fontsize=14)
axes[1].set_ylabel('Exam Score')

plt.tight_layout()
plt.savefig('../data/processed/eda_exam_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📈 Estadísticas Exam_Score:")
print(f"  Media:   {df['Exam_Score'].mean():.2f}")
print(f"  Mediana: {df['Exam_Score'].median():.2f}")
print(f"  Std:     {df['Exam_Score'].std():.2f}")
print(f"  Min:     {df['Exam_Score'].min()}")
print(f"  Max:     {df['Exam_Score'].max()}")

## 3. Análisis de Variables Numéricas

In [ ]:
# Identificar columnas numéricas
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols.remove('Exam_Score')  # Excluir target

print(f"📊 Variables numéricas ({len(num_cols)}):")
for col in num_cols:
    print(f"  - {col}: [{df[col].min()}, {df[col].max()}], "
          f"mean={df[col].mean():.1f}")

In [ ]:
# Distribuciones de variables numéricas
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols[:6]):
    axes[i].hist(df[col], bins=25, edgecolor='black', alpha=0.7, color='coral')
    axes[i].set_title(f'{col}', fontsize=12)
    axes[i].set_xlabel('')

# Ocultar ejes vacíos
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribuciones de Variables Numéricas', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/eda_numerical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Matriz de correlación
corr_matrix = df[num_cols + ['Exam_Score']].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Matriz de Correlación', fontsize=16)
plt.tight_layout()
plt.savefig('../data/processed/eda_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top correlaciones con Exam_Score
corr_with_target = corr_matrix['Exam_Score'].drop('Exam_Score').abs().sort_values(ascending=False)
print("🔗 Top correlaciones con Exam_Score:")
for feat, corr in corr_with_target.head(10).items():
    direction = "↑" if corr_matrix.loc[feat, 'Exam_Score'] > 0 else "↓"
    print(f"  {direction} {feat}: {corr:.3f}")

## 4. Análisis de Variables Categóricas

In [ ]:
# Identificar columnas categóricas
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"📊 Variables categóricas ({len(cat_cols)}):")
for col in cat_cols:
    n_unique = df[col].nunique()
    print(f"  - {col}: {n_unique} categorías → {df[col].unique()[:5]}")

In [ ]:
# Distribución de variables categóricas
fig, axes = plt.subplots(3, 4, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    value_counts = df[col].value_counts()
    axes[i].barh(value_counts.index, value_counts.values, color='teal', alpha=0.7)
    axes[i].set_title(f'{col}', fontsize=11)
    axes[i].set_xlabel('Count')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribuciones de Variables Categóricas', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/eda_categorical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Relaciones Categóricas vs Target

In [ ]:
# Boxplots de Exam_Score por variable categórica
key_cat_cols = ['Parental_Involvement', 'Motivation_Level', 'School_Type', 
                'Internet_Access', 'Teacher_Quality', 'Family_Income']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(key_cat_cols):
    sns.boxplot(data=df, x=col, y='Exam_Score', ax=axes[i], palette='Set2')
    axes[i].set_title(f'Exam_Score vs {col}', fontsize=12)
    axes[i].tick_params(axis='x', rotation=30)

plt.suptitle('Relación Variables Categóricas vs Exam_Score', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/eda_cat_vs_target.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Análisis Bivariado

In [ ]:
# Scatter plots de las features más correlacionadas
top_features = corr_with_target.head(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    axes[i].scatter(df[feat], df['Exam_Score'], alpha=0.4, s=20, color='navy')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Exam_Score')
    axes[i].set_title(f'{feat} vs Exam_Score')
    
    # Línea de tendencia
    z = np.polyfit(df[feat], df['Exam_Score'], 1)
    p = np.poly1d(z)
    axes[i].plot(df[feat], p(df[feat]), "r--", alpha=0.8, linewidth=2)

plt.suptitle('Relaciones Bivariadas con Exam_Score', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/eda_bivariate.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Resumen y Conclusiones

In [ ]:
print("=" * 60)
print("📋 RESUMEN DEL ANÁLISIS EXPLORATORIO")
print("=" * 60)
print(f"\n📐 Dataset: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"🎯 Target: Exam_Score (range: {df['Exam_Score'].min()}-{df['Exam_Score'].max()})")
print(f"📊 Features numéricas: {len(num_cols)}")
print(f"📊 Features categóricas: {len(cat_cols)}")
print(f"🔍 Valores nulos: {df.isnull().sum().sum()}")
print(f"\n🔗 Top 5 features correlacionadas con target:")
for i, (feat, corr) in enumerate(corr_with_target.head(5).items(), 1):
    print(f"   {i}. {feat}: {corr:.3f}")
print(f"\n✅ EDA completado. Gráficos guardados en data/processed/")
print("=" * 60)